# Defining Tools
### allows Frontier models to connect with external functions
Richer responses by extending knowledge

Ability to cary out actions within the application

Enhanced capabilities, like calculations

# Building an Airline AI Assistant with Tool Calling in OpenAI and Gradio

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
import gradio as gr
load_dotenv(override=True)
from openai import OpenAI

In [2]:
API_KEY = os.getenv("API_KEY")

In [3]:
API_KEY[:6]

'sk-pro'

In [4]:
MODEL = "gpt-4.1-mini"
openai = OpenAI(api_key=API_KEY)

In [5]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [6]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_message}]+history+[{"role":"user","content":message}]
    response = openai.chat.completions.create(model=MODEL,messages=messages)
    return response.choices[0].message.content

# Making gradio Interface
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [17]:
# Let's start by making a useful function

ticket_prices = {"london":"$799","paris":"$899","tokyo":"$1400","berlin":"$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(),"Unknown Ticket Price")
    return f"The price of ticket to {destination_city} is {price}"

In [18]:
get_ticket_price("Berlin")

Tool called for city Berlin


'The price of ticket to Berlin is $499'

In [19]:
get_ticket_price("Belin")

Tool called for city Belin


'The price of ticket to Belin is Unknown Ticket Price'

In [20]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [21]:
# and this is included in a list of tools
tools = [{"type":"function","function":price_function}]

In [22]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [23]:
# Getting OpenAI to use our Tool

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [24]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls": # means the AI model stopped generating text because it wants to run an external function or tool instead of giving a final text answer. 
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [25]:
# Lets write handle_tool_call function

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city= arguments.get("destination_city")
        price_details = get_ticket_price(city)
        response={
            "role":"tool",
            "content":price_details,
            "tool_call_id":tool_call.id
        }
    return response

In [26]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls": # means the AI model stopped generating text because it wants to run an external function or tool instead of giving a final text answer. 
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content

In [27]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [28]:
# when we call a tool for two options we get an error 

let us make a couple of improvements
 handling multiple tool calls in 1 response
 handling multiple tool calls 1 after another

In [29]:
# Lets write handle_tool_call function

def handle_tool_call(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city= arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role":"tool",
                "content":price_details,
                "tool_call_id":tool_call.id
            })
    return responses

In [30]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls": # means the AI model stopped generating text because it wants to run an external function or tool instead of giving a final text answer. 
        message = response.choices[0].message
        responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content

In [31]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [32]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls": # means the AI model stopped generating text because it wants to run an external function or tool instead of giving a final text answer. 
        message = response.choices[0].message
        responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content

In [33]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


# Building Tool calling SQLite Database Integeration

In [34]:
%pip install sqlite3

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


In [35]:
import sqlite3

In [36]:
help(sqlite3)

Help on package sqlite3:

NAME
    sqlite3

MODULE REFERENCE
    https://docs.python.org/3.14/library/sqlite3#module-sqlite3

    The following documentation is automatically generated from the Python
    source files.  It may be incomplete, incorrect or include features that
    are considered implementation detail and may vary between Python
    implementations.  When in doubt, consult the module reference at the
    location listed above.

DESCRIPTION
    The sqlite3 extension module provides a DB-API 2.0 (PEP 249) compliant
    interface to the SQLite library, and requires SQLite 3.15.2 or newer.

    To use the module, start by creating a database Connection object:

        import sqlite3
        cx = sqlite3.connect("test.db")  # test.db will be created or opened

    The special path name ":memory:" can be provided to connect to a transient
    in-memory database:

        cx = sqlite3.connect(":memory:")  # connect to a database in RAM

    Once a connection has been establish

In [37]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY,price REAL)")
    conn.commit()

In [38]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?",(city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available"

In [39]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [40]:
def set_ticket_price(city,price):
    cursor = conn.cursor()
    cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
    conn.commit()

In [41]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city,price in ticket_prices.items():
    set_ticket_price(city,price)

In [42]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $799.0'

In [43]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls": # means the AI model stopped generating text because it wants to run an external function or tool instead of giving a final text answer. 
        message = response.choices[0].message
        responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content

In [44]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Exercise

Add a tool to set the price of a ticket!

In [45]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls": # means the AI model stopped generating text because it wants to run an external function or tool instead of giving a final text answer. 
        message = response.choices[0].message
        responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content

In [46]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to a specified destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The target destination city (e.g., 'London', 'Tokyo')",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price amount",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False,
    },
}

In [47]:
# and this is included in a list of tools
tools = [{"type":"function","function":price_function},
         {"type":"function","function":set_price_function}]

In [48]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_ticket_price',
   'description': 'Set the price of a return ticket to a specified destination city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': "The target destination city (e.g., 'London', 'Tokyo')"},
     'price': {'type': 'number',
      'description': 'The new ticket price amount'}},
    'required': ['city', 'price'],
    'additionalProperties': False}}}]

In [49]:
def handle_tool_call(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city= arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role":"tool",
                "content":price_details,
                "tool_call_id":tool_call.id
            })
        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city= arguments.get("destination_city")
            price_details = get_ticket_price(city)
            new_price = arguments.get("new_price")
            responses.append({
                "role":"tool",
                "content":set_ticket_price,
                "tool_call_id":tool_call.id
            })
    return responses

In [50]:
def set_ticket_price(city,price):
    print("tool called set price for "+city)
    cursor = conn.cursor()
    cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
    conn.commit()
    return f"Ticket price to {city} is set to ${price}" if result else "No price data available for this city"

In [51]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [52]:
can you set paris price to $50

SyntaxError: invalid syntax (3244774063.py, line 1)

### How Function Calling Works (The Core Concept)
The AI model itself cannot touch your local database or run Python code directly.

Instead, you send the AI a list of "tools" (descriptions of what your functions do).

When you ask a question like "Change the ticket price for London to $500", the AI notices it has a tool named set_ticket_price.

Instead of answering with plain text, the AI replies with a request: "Please run set_ticket_price with city='London' and price=500."

Your Python script executes that function locally, interacts with SQLite, and sends the result back to the AI.

The AI reads the result and gives a final friendly answer to the use

In [55]:
import json

# defining the tool json, This is an Instruction Manual for the AI 
# You tell the ai the name of the tool set_ticket_price
# its description and its inputs
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to a specified destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The target destination city (e.g., 'London', 'Tokyo')",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price amount",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False,
    },
}
# we defined the tools as dictionaries of these json files
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]
# Required for connecting to DB
import sqlite3

DB_PATH = "prices.db"  # Path to your SQLite database file

def set_ticket_price(city, price):
    print("tool called set price for " + city)
    
    # Create a connection inside the active thread
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
            (city.lower(), price, price)
        )
        conn.commit()
        
    return f"Ticket price to {city} is set to ${price}"


def handle_tool_call(message):
    responses = []
    #Purpose: Creates an empty list to store the execution results. If the AI asks to run multiple tools at once 
    #(e.g., getting prices for both London and Tokyo in a single prompt), this list will collect all the answers.
    
    for tool_call in message.tool_calls: # a list of tool requests sent by ai
        arguments = json.loads(tool_call.function.arguments)
        #The AI sends function parameters as a JSON string 
        #(e.g., '{"city": "London", "price": 500}').

        # checks which tool is being called
        if tool_call.function.name == "get_ticket_price":
            city = arguments.get("city") or arguments.get("destination_city")
            #Pulls the city name out of the arguments dictionary 
            #(arguments.get("city")). 
            #The or acts as a fallback if the key was named destination_city.
            
            price_details = get_ticket_price(city) #calls your get_ticekt price
            responses.append({
                "role": "tool", # Tells OpenAi this message is a tool
                "content": str(price_details), #The output from your function, converted to a string.
                "tool_call_id": tool_call.id # Tool id
            })
            
        elif tool_call.function.name == "set_ticket_price":
            city = arguments.get("city") # Extracting variables from the dictionary
            price = arguments.get("price")
            
            # Execute the function with parsed parameters
            result = set_ticket_price(city, price)
            
            responses.append({
                "role": "tool",
                "content": str(result),
                "tool_call_id": tool_call.id
            })
            
    return responses


def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        responses = handle_tool_call(msg)
        messages.append(msg)
        messages.extend(responses)
        # Pass tools back into subsequent API calls so parallel/multi-step tool calls work
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [56]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Paris
tool called set price for Paris


In [58]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $100.0'

DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
